# Two-Tower Model (TTN) — dataset

Assembles the dataset the two-tower model trains on, and splits it in time.

Three sources, one row per interaction:

| Source | Grain | Joined how |
| --- | --- | --- |
| `Home_and_Kitchen_filtered.csv` | one row per review | the base table |
| `df_features.pkl` | one row per `asin` | left join on `asin` |
| `df_user_features.pkl` | one row per purchase event | positional concat |

Model definition and training come next, once the columns are settled — §6 is
where that picks up.

In [71]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the user–item interactions

`data/Home_and_Kitchen_filtered.csv` is the interaction log: **one row per
review**, i.e. one row per (user, item, date) event. This is the table the
two-tower model is trained on — every other dataset in this notebook is joined
onto it.

**All 11 columns are loaded here** so the full contents are visible before
anything is thrown away. The next section decides what to keep.

`asin` and `reviewerID` are pinned to `str` so IDs with leading zeros
(e.g. `0560467893`) survive, and `low_memory=False` avoids the mixed-type
warning on `vote`.

In [72]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

df_reviews = pd.read_csv(
    DATA_DIR / "Home_and_Kitchen_filtered.csv",
    dtype={"asin": str, "reviewerID": str},
    low_memory=False,
)

print("df_reviews:", df_reviews.shape)
print("columns:", list(df_reviews.columns))
print(f"unique users: {df_reviews['reviewerID'].nunique():,} | "
      f"unique items: {df_reviews['asin'].nunique():,}")
df_reviews.head(5)

df_reviews: (6898955, 11)
columns: ['overall', 'verified', 'reviewTime', 'reviewerID', 'asin', 'reviewerName', 'summary', 'unixReviewTime', 'vote', 'style', 'image']
unique users: 777,242 | unique items: 189,172


,overall,verified,reviewTime,reviewerID,asin,reviewerName,summary,unixReviewTime,vote,style,image
0,5.00,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,Linda Fahner,Five Stars,1446681600,NaN,NaN,NaN
1,3.00,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,Harry Slaughter,Meh,1430956800,2,NaN,NaN
2,5.00,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,luckyg,Recommend,1390348800,NaN,{'Color:': ' Brushed Stainless'},NaN
3,1.00,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,Nickleen,Not keeping coffee hot for long enough,1383091200,NaN,{'Color:': ' Brushed Stainless'},NaN
4,1.00,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,Lacemaker427,Leaks like a waterfall when at an angle!,1379635200,NaN,{'Color:': ' Red'},NaN


### Drop what the model cannot use at serving time

Two reasons to drop a column, and it matters which applies:

**(a) Post-interaction.** `overall`, `vote`, `summary`, `image` only exist
*after* the purchase. At recommendation time we don't have them, so training on
them learns from information that will never be there in production. Training
here is implicit — the interaction itself is the positive signal — so the rating
is neither needed as a label nor usable as an input.

**(b) No signal.** `reviewerName` is a display name, not an identifier;
`reviewerID` already identifies the user. `style` describes a *variant* while
the model recommends at `asin` level, and it is a high-cardinality dict string.

Kept: `reviewerID` + `asin` (the interaction), `unixReviewTime` (the timeline
and the split key), `reviewTime` (readable duplicate), `verified` (a property of
the transaction, so known at interaction time).

⚠️ `style` and `summary` are what separate a genuine same-day purchase of two
variants from the same review recorded twice. If you de-duplicate the log, do it
**before** this cell — 231,139 rows are exact duplicates across all 11 columns,
but deduping after this drop collapses ~17k real interactions too.

In [74]:
# (a) post-interaction — known only after the purchase, would leak at serving time
POST_INTERACTION = ["overall", "vote", "summary", "image"]

# (b) no signal / redundant
NO_SIGNAL = ["reviewerName", "style"]

DROP_COLS = POST_INTERACTION + NO_SIGNAL

df_reviews = df_reviews.drop(columns=[c for c in DROP_COLS if c in df_reviews.columns])

print(f"dropped {len(DROP_COLS)}: {DROP_COLS}")
print(f"kept    {df_reviews.shape[1]}: {list(df_reviews.columns)}")
print("\ndf_reviews:", df_reviews.shape)
df_reviews.head(5)

dropped 6: ['overall', 'vote', 'summary', 'image', 'reviewerName', 'style']
kept    5: ['verified', 'reviewTime', 'reviewerID', 'asin', 'unixReviewTime']

df_reviews: (6898955, 5)


,verified,reviewTime,reviewerID,asin,unixReviewTime
0,True,"11 5, 2015",A8LUWTIPU9CZB,0560467893,1446681600
1,True,"05 7, 2015",A3B6GKQQ1JJ167,0560467893,1430956800
2,True,"01 22, 2014",A3MCTN65BU7XRA,0681795107,1390348800
3,True,"10 30, 2013",A7JVZFSXVY9RL,0681795107,1383091200
4,True,"09 20, 2013",A2RQ7VLAK1SHPU,0681795107,1379635200


## 2. Item features

`data/df_features.pkl` — one row per `asin` (~1.13M items) with the attributes
extracted by `feature_extraction_workflow/extract_features.py`: `cat_*`,
`brand`, the per-field columns (`Product_Type`, `Material`, `Color`, …), their
parsed `_numeric` / `_unit` / `_cleaned` measures, and `title_cleaned`.

It is loaded and **analysed on its own first**. The join onto the interactions
happens in §4, once the checks below have settled which columns are worth
carrying.

In [75]:
from pathlib import Path

# data/ lives at the repo root, one level up from this ttn/ folder
DATA_DIR = Path("..") / "data"

# --- Item features (one row per asin) ---
df_features = pd.read_pickle(DATA_DIR / "df_features.pkl")
print("df_features:", df_features.shape)
df_features.head(3)

df_features: (1134566, 82)


,category,tech1,description,title,tech2,brand,feature,rank,main_cat,price,asin,date,imageURL,imageURLHighRes,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6,title_cleaned,extracted_features_title,description_cleaned,extracted_features_description,feature_cleaned,extracted_features_feature,extracted_features,Bar_Pressure,Brand,Capacity_Cups,Capacity_Volume,Color,Density_Weight,Dimensions,Features,Filter_Rating,Material,Part_Number,Piece_Count,Pocket_Depth,Power_Rating,Product_Type,Scent,Shape,Shape_Style,Size,Stage_Count,Sub_Type,Theme,Thread_Count,Voltage,Weight,capacity_volume_numeric,capacity_volume_unit,piece_count_numeric,piece_count_unit,thread_count_numeric,thread_count_unit,weight_numeric,weight_unit,bar_pressure_numeric,capacity_cups_numeric,density_weight_lb,pocket_depth_in,power_rating_w,stage_count_numeric,voltage_numeric,dimension_1,dimension_2,dimension_3,dimension_unit,bar_pressure_numeric_cleaned,capacity_cups_numeric_cleaned,density_weight_lb_cleaned,pocket_depth_in_cleaned,power_rating_w_cleaned,stage_count_numeric_cleaned,voltage_numeric_cleaned,capacity_volume_numeric_cleaned,weight_numeric_cleaned,piece_count_numeric_cleaned,thread_count_numeric_cleaned
0,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,['It was a time honored tradition among the ea...,You Are Special Today Red Plate [With Red Pen],NaN,Waechtersbach USA,[],"['>#39,665 in Kitchen & Dining (See Top 100 in...",Amazon Home,$37.00,0001487795,"October 8, 2006",[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Dinnerware,Plates,Dinner Plates,special today red plate red pen,{'Color': 'red'},time honored tradition among early american fa...,{'Color': 'red'},,{},{'Color': 'red'},None,None,None,None,red,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"['Home & Kitchen', 'Home Dcor', 'Candles & Hol...",NaN,['VICKS INHALER relieves stuffy noses helps si...,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,NaN,Vicks,[],"['>#1,763,185 in Home & Kitchen (See Top 100 i...",Amazon Home,$4.05,0002020300,NaN,[],[],Home & Kitchen,Home Dcor,Candles & Holders,Candles,None,None,vicks inhaler relief cold sinus nasal congesti...,{},vicks inhaler relief stuffy nose help sinus co...,{},,{},{},None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"['Home & Kitchen', 'Kitchen & Dining', 'Dining...",NaN,"['16 oz squeeze bottle, 1 lb.']",Artistic Churchware Communion Cup Filler: RW525,NaN,Artistic Churchware,"['Religious Supply Center', 'RW-525', 'Communi...","['>#2,127,003 in Home & Kitchen (See Top 100 i...",Amazon Home,$12.48,0006564224,NaN,[],[],Home & Kitchen,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Wine & Champagne Glasses,None,artistic churchware communion cup filler rw525,{'Product_Type': 'cup'},16 oz squeeze bottle 1 lb,{'Capacity_Volume': '16 oz'},religious supply center rw-525 communion cup f...,{'Product_Type': 'cup'},"{'Product_Type': 'cup', 'Capacity_Volume': '16...",None,None,None,16 oz,None,None,None,None,None,None,None,None,None,None,cup,None,None,None,None,None,None,None,None,None,None,16.00,oz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.00,NaN,NaN,NaN


### Validate the feature table

Before anything is joined, check that `df_features.pkl` is what the pipeline
promised: the exact expected column set, one row per `asin`, numeric columns
actually numeric, coverage above its floors, unit columns free of new values,
and every `_cleaned` column inside its bound.

A **FAIL on `columns`** is the one to care about most — it means a feature
appeared that nothing describes, or one silently disappeared.

In [ ]:
import sys
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from feature_extraction_workflow.validations import run_all

report = run_all(df_features, DATA_DIR / "master_metadata.json")
display(report if len(report) else "no findings")

### Two cleaning steps that still live here

`cat_4_clean` is built from `data/category_taxonomy.json` — a reviewed
whitelist of which `(cat_3, cat_4)` pairs are real categories rather than
product bullets that leaked into the path. 921 → 451 distinct values, with
0.1% of items landing in a `<cat_3>_Other` bucket. §3 joins `cat_4_clean`
rather than raw `cat_4`.

`brand_clean` folds brands carried by ten or fewer items into `other_brands`.

Both are also produced by the pipeline now (`clean_brand` is Filter 5), so once
you re-extract, the brand cell here is recomputing what the pickle already
carries.

In [77]:
import json

TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

# cat_3 -> the set of cat_4 values that survived the review
valid_cat_4 = {c3: set(vals) for c2 in taxonomy for c3, vals in taxonomy[c2].items()}
print(f"taxonomy: {len(taxonomy)} cat_2 | {len(valid_cat_4)} cat_3 | "
      f"{sum(len(v) for v in valid_cat_4.values())} valid cat_4 slots")

MISSING = "Missing"
OTHER_SUFFIX = "_Other"

cat_3 = df_features["cat_3"].astype(str)
cat_4 = df_features["cat_4"].fillna(MISSING).astype(str)

# A value is kept only if it is valid *under its own parent* — the same label
# can be real in one branch and junk in another.
valid_pairs = {(c3, v) for c3, vals in valid_cat_4.items() for v in vals}
keep = pd.Series(list(zip(cat_3, cat_4)), index=df_features.index).isin(valid_pairs)

df_features["cat_4_clean"] = np.where(keep, cat_4, cat_3 + OTHER_SUFFIX)

n_before = df_features["cat_4"].nunique(dropna=False)
n_after = df_features["cat_4_clean"].nunique()
n_folded = int((~keep).sum())
print(f"\ndistinct cat_4 : {n_before:,} -> {n_after:,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features) * 100:.2f}% of the catalog)")
df_features[["asin", "cat_2", "cat_3", "cat_4", "cat_4_clean"]].head(5)

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid cat_4 slots

distinct cat_4 : 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10% of the catalog)


,asin,cat_2,cat_3,cat_4,cat_4_clean
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,Dinnerware
1,0002020300,Home Dcor,Candles & Holders,Candles,Candles
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,Glassware & Drinkware
3,0009046461,Bath,Bathroom Accessories,None,Missing
4,0234937912,Home Dcor,Home Fragrance,Incense & Incense Holders,Incense & Incense Holders


In [ ]:
# Fold rare brands: keep those carried by MORE THAN 10 distinct items.
# Counted on brand_norm (from 3.4), not the raw column — otherwise "3d rose" and
# "3drose" are counted separately and a brand can fall under the threshold only
# because its spelling is split.
MIN_ITEMS = 10
OTHER = "other_brands"

source = "brand_norm" if "brand_norm" in df_features.columns else "brand"
brand_counts = df_features.groupby(source)["asin"].nunique()

kept = brand_counts[brand_counts > MIN_ITEMS].index
df_features["brand_clean"] = df_features[source].where(
    df_features[source].isin(kept) | df_features[source].isna(),
    OTHER,
)

n_before = df_features[source].nunique()
n_after = df_features["brand_clean"].nunique()
n_missing = df_features[source].isna().sum()
print(f"counted on : {source}")
print(f"brands     : {n_before:,} -> {n_after:,} "
      f"(kept {len(kept):,} with > {MIN_ITEMS} items, rest -> {OTHER!r})")
print(f"items       : {(df_features['brand_clean'] == OTHER).mean():.1%} in {OTHER}, "
      f"{n_missing / len(df_features):.1%} missing (left as NaN)")
print()
print(df_features["brand_clean"].value_counts().head(20))

# --- Choosing the threshold ------------------------------------------------
# The cut above keeps brands carried by >= 10 items. What the model actually
# sees is INTERACTIONS, though: a brand on 3 items with 8,000 reviews has
# plenty of signal, while one on 40 items with 45 reviews gives its embedding
# ~1 gradient update per row. Judge a threshold by the share of the data that
# keeps a real brand, not by how small the table gets.

def brand_coverage(counts, label, thresholds=(1, 5, 10, 20, 50, 100, 500, 1000)):
    total_units, total_brands = counts.sum(), len(counts)
    rows = []
    for n in thresholds:
        keep = counts[counts >= n]
        rows.append({"min_" + label: n,
                     "brands_kept": len(keep),
                     "brands_kept_%": len(keep) / total_brands * 100,
                     f"{label}_covered_%": keep.sum() / total_units * 100})
    return pd.DataFrame(rows).set_index("min_" + label)

print("\nby ITEMS (what df_features can measure):")
display(brand_coverage(brand_counts, "items").round(2))

# The interaction view needs the merged frame from section 4; run this again
# after the join to pick the threshold on the number that matters.
if "df" in dir() and "brand" in getattr(df, "columns", []):
    print("by INTERACTIONS (post-join):")
    display(brand_coverage(df.groupby("brand").size(), "interactions").round(2))
else:
    print("by INTERACTIONS: run again after the join in section 4 "
          "(`df` not built yet)")

## 3. Join the item features onto the interactions

With the examination above settled, attach the item columns to the interaction
table on `asin`. Left join, so no interaction is dropped: items with no
extracted features keep their row with NaN item columns (~16% of rows —
those asins either aren't in the item metadata, or were dropped during
extraction because their `cat_3` has no schema).

The cell below currently attaches **every** field and measure column. Once §3
has produced a keep-list, narrow `keep_cols` to it — that is the one place the
exclusion decision needs to be applied.

In [ ]:
# --- Connect the two on `asin` ---
# Attach ALL extracted feature variables plus their parsed measures.

# 1. Every extracted feature field column (the keys present in the dicts):
#    Product_Type, Material, Color, Weight, Dimensions, Brand, Theme, ...
field_cols = sorted({
    k for d in df_features["extracted_features"]
    if isinstance(d, dict) for k in d
})

# 2. Their parsed measures: numeric value + unit (+ dimension_1/2/3).
#    Use the range-cleaned numeric variant wherever one exists.
measure_cols = [
    c for c in df_features.columns
    if c.endswith("_numeric") or c.endswith("_unit")
    or c.startswith("dimension_")
    or c in ("density_weight_lb", "pocket_depth_in", "power_rating_w")
]
measure_cols = sorted({
    f"{c}_cleaned" if f"{c}_cleaned" in df_features.columns else c
    for c in measure_cols
})

# 3. Context columns to carry along (asin is the join key).
# cat_4_clean (from category_taxonomy.json) replaces the raw cat_4 here —
# the ~920 raw values are mostly bullet text, see section 3.
context_cols = ["asin", "cat_2", "cat_3", "cat_4_clean", "brand", "extracted_features"]

keep_cols = list(dict.fromkeys(context_cols + field_cols + measure_cols))
keep_cols = [c for c in keep_cols if c in df_features.columns]

df = df_reviews.merge(
    df_features[keep_cols],
    on="asin",
    how="left",
    validate="many_to_one",   # many reviews -> one item row
    indicator=True,
)
n_unmatched = (df["_merge"] == "left_only").sum()
df = df.drop(columns="_merge")

print(f"merged: {df.shape}  ({len(keep_cols)} item cols attached)")
print(f"  feature fields : {len(field_cols)}")
print(f"  measure cols   : {len(measure_cols)}")
print(f"unique users: {df['reviewerID'].nunique():,} | "
      f"unique items: {df['asin'].nunique():,}")
print(f"reviews with no matching item features: {n_unmatched:,}")
df.head(5)

## 4. User features

`data/df_user_features.pkl` — built by
`feature_extraction_workflow/extract_features_user.py`. **One row per purchase
event** with 22 features, each computed only from that user's purchases on
**strictly earlier days**, so no row can see its own or any later interaction.
See that module's section in `feature_extraction_workflow/README.md`.

### Why this is a positional concat, not a merge

The natural key is (user, item, date) — joining on `reviewerID` alone would be a
real leak, attaching every event's feature vector, including future ones, to
every row of that user.

But that triple **isn't unique in this log**: 501,704 rows share a
`(reviewerID, asin, unixReviewTime)` triple with at least one other row, so a
key merge is many-to-many and square-joins those groups into **+572,198 phantom
rows (+8.3%)**.

The pickle was generated from this exact CSV, in file order, dropping nothing —
so row *i* of the pickle already is row *i* of the log. Aligning by position
gives identical semantics with zero fanout.

The asserts below **verify** that alignment element-wise instead of assuming it.
They also confirm the row order survived the item join in §4. If the pickle is
ever regenerated from a different or reordered log, this fails loudly rather
than silently pairing the wrong user's history to a row.

**Expect ~23.5% of rows to have all-NaN user features.** That's a user's first
purchase (and anything on that same first day) — there is no prior history to
summarize. Intended, not a defect.

In [ ]:
df_user_features = pd.read_pickle(DATA_DIR / "df_user_features.pkl")
print("df_user_features:", df_user_features.shape)
display(df_user_features.head(3))

# --- Verify positional alignment before relying on it ---
assert len(df_user_features) == len(df), (
    f"row count mismatch: user features {len(df_user_features):,} vs "
    f"interactions {len(df):,}"
)
for key in ("reviewerID", "asin", "unixReviewTime"):
    n_bad = int((df_user_features[key].to_numpy() != df[key].to_numpy()).sum())
    assert n_bad == 0, f"row order mismatch on {key}: {n_bad:,} rows differ"
print("alignment verified: reviewerID / asin / unixReviewTime match row-for-row\n")

# Attach the 22 feature columns (the key columns are already in df)
USER_FEATURE_COLS = [
    c for c in df_user_features.columns
    if c not in ("reviewerID", "asin", "reviewTime", "unixReviewTime")
]
df = pd.concat(
    [df.reset_index(drop=True),
     df_user_features[USER_FEATURE_COLS].reset_index(drop=True)],
    axis=1,
)

n_cold = int(df["prior_purchase_count"].isna().sum())
print(f"attached {len(USER_FEATURE_COLS)} user feature columns -> df {df.shape}")
print(f"cold-start rows (empty history, all user features NaN): "
      f"{n_cold:,} ({n_cold / len(df) * 100:.1f}%)")
df.head(5)

## 5. Every column in the final dataset

`df` is now the full table: one row per interaction, carrying the interaction
keys, the item features joined on `asin`, and the user features aligned by
position.

The inventory below is the list to work from when deciding what the towers
actually consume — `source`, `dtype`, coverage, distinct count and a few example
values per column. `source` is usually the first thing that settles a column's
fate: interaction columns are known at serving time, item columns describe the
catalogue, user columns are the leakage-safe history.

In [ ]:
# --- Every column in the final dataset, with what you need to judge it ------
# `source` says where a column came from, which is usually the first thing that
# decides its fate: an interaction column is known at serving time, an item
# column describes the catalogue, a user column is the leakage-safe history.

INTERACTION_COLS = set(df_reviews.columns)
USER_COLS = set(USER_FEATURE_COLS)


def _source(col):
    if col in USER_COLS:
        return "user"
    if col in INTERACTION_COLS:
        return "interaction"
    return "item"


def column_inventory(frame=None, sample=3):
    frame = df if frame is None else frame
    rows = []
    for c in frame.columns:
        s = frame[c]
        nn = int(s.notna().sum())
        try:
            distinct = int(s.nunique(dropna=True))
        except TypeError:                      # dict/list columns
            distinct = np.nan
        try:
            vals = s.dropna().unique()[:sample]
            example = ", ".join(str(v)[:28] for v in vals)
        except Exception:
            example = ""
        rows.append({
            "column": c,
            "source": _source(c),
            "dtype": str(s.dtype),
            "non_null": nn,
            "coverage_%": nn / len(frame) * 100,
            "distinct": distinct,
            "example": example[:64],
        })
    out = (pd.DataFrame(rows)
             .sort_values(["source", "coverage_%"], ascending=[True, False])
             .reset_index(drop=True))
    return out


inventory = column_inventory()
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
print(inventory.groupby("source")["column"].count().to_string(), "\n")
display(inventory)

## 6. Train / test split

Same convention as `functions/interaction_matrix.py` and `adding_bpr.ipynb`: one
global time cutoff, train strictly before it, test at or after, and items kept
only where at least `MIN_USERS` distinct users interacted with them.

Two properties worth keeping:

- **A global cutoff, not a per-user one.** Every training row precedes every
  test row, so nothing in train can depend on a future the model would not have
  had.
- **The item filter is fitted on train only.** Choosing which items exist using
  test rows would leak the evaluation period into the vocabulary — the same rule
  the user-feature pipeline follows.

Transformations you want on the features (encoding, bucketing, scaling) belong
between §5 and here, so they can be fitted on `train` alone.

In [ ]:
# --- Train / test split ----------------------------------------------------
# Same convention as `functions/interaction_matrix.py` and adding_bpr.ipynb:
# a single global time cutoff, train strictly before it, test at or after it,
# and items kept only if at least `MIN_USERS` distinct users interacted with
# them. A global cutoff rather than a per-user one keeps the split honest —
# every training row precedes every test row, so nothing in train can depend on
# a future the model would not have had.

MIN_USERS = 5           # matches InteractionMatrixBuilder's default
TEST_FRACTION = 0.1     # size of the held-out tail

cutoff_time = df["unixReviewTime"].quantile(1 - TEST_FRACTION)
print(f"cutoff: {pd.to_datetime(cutoff_time, unit='s').date()} "
      f"(unix {int(cutoff_time):,})")

# Item filter is fitted on TRAIN ONLY — deciding which items exist using test
# rows would leak the evaluation period into the vocabulary.
train = df[df["unixReviewTime"] < cutoff_time].copy()
test = df[df["unixReviewTime"] >= cutoff_time].copy()

item_users = train.groupby("asin")["reviewerID"].nunique()
keep_items = item_users[item_users >= MIN_USERS].index
train = train[train["asin"].isin(keep_items)]

# A test row is only scoreable if both its user and its item were seen in train.
test = test[test["asin"].isin(keep_items) & test["reviewerID"].isin(train["reviewerID"].unique())]

print(f"\ntrain : {len(train):>10,} rows | {train['reviewerID'].nunique():>8,} users "
      f"| {train['asin'].nunique():>7,} items")
print(f"test  : {len(test):>10,} rows | {test['reviewerID'].nunique():>8,} users "
      f"| {test['asin'].nunique():>7,} items")
print(f"\nitems dropped by the >= {MIN_USERS}-user filter: "
      f"{df['asin'].nunique() - len(keep_items):,} of {df['asin'].nunique():,}")
print(f"test rows dropped as unscoreable (cold user or item): "
      f"{len(df[df['unixReviewTime'] >= cutoff_time]) - len(test):,}")